In [0]:
import requests
import json
import os
from datetime import datetime, timedelta, timezone

catalog = "openaq"
schema = "livestreamdata"

api_key = dbutils.secrets.get(scope="openaq_project", key="OpenAQ_api_key")
headers = {"X-API-Key": api_key}

locations = {
    "san_francisco": "2009",
    "las_vegas_casino_center": "7712",
    "boston_roxbury": "448"
}

date_to = datetime.now(timezone.utc)
date_from = date_to - timedelta(days=1)
timestamp = date_to.strftime("%Y%m%d_%H%M%S")

for folder_name, loc_id in locations.items():
    loc_url = f"https://api.openaq.org/v3/locations/{loc_id}"
    loc_res = requests.get(loc_url, headers=headers).json()
    
    sensors = loc_res.get('results', [{}])[0].get('sensors', [])
    daily_measurements = []
    
    for sensor in sensors:
        sensor_id = sensor.get('id')
        meas_url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements"
        
        params = {
            "date_from": date_from.strftime("%Y-%m-%dT%H:%M:%S"),
            "limit": 1000
        }
        
        meas_res = requests.get(meas_url, params=params, headers=headers).json()
        daily_measurements.extend(meas_res.get('results', []))
    
    volume_path = f"/Volumes/{catalog}/{schema}/raw_data/openaq_v3/{folder_name}"
    file_name = f"{volume_path}/data_daily_{timestamp}.json"
    
    os.makedirs(volume_path, exist_ok=True)
    with open(file_name, "w") as f:
        json.dump({"results": daily_measurements}, f)
    
    print(f"Ingesta Diaria - {folder_name} | Registros: {len(daily_measurements)} | Archivo: {file_name}")